In [ ]:
from google.colab import files
files.upload()

In [ ]:
!ls

In [ ]:
!mkdir -p ~/.kaggle
!cp "kaggle (1).json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets download -d mariaherrerot/aptos2019

In [ ]:
!ls

In [ ]:
!unzip aptos2019.zip

In [ ]:
!ls

In [ ]:
!git clone https://github.com/haliait/sys_retinopathie_diabetique

In [ ]:
%cd sys_retinopathie_diabetique

In [ ]:
!ls

In [ ]:
!git pull origin main

In [ ]:
!git config --global user.name "SaraTaha9"
!git config --global user.email "tsara6317@email.com"

# Imports & configuration

In [ ]:
import kagglehub
path = kagglehub.dataset_download("mariaherrerot/aptos2019")

In [ ]:
print(path)

In [ ]:
import os
os.listdir(path)

In [ ]:
# ============================================================
# SECTION 1 — EXPLORATION DES DONNÉES (EDA)
# Projet PFA : Détection intelligente de la rétinopathie diabétique
# Dataset   : APTOS 2019 Blindness Detection
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import os
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Reproductibilité — obligatoire pour un projet académique
np.random.seed(42)

# ── Chemins Kaggle ──────────────────────────────────────────
# Vérifier d'abord avec : os.listdir('/kaggle/input/')
BASE_PATH = path         # adapter si besoin
TRAIN_CSV  = os.path.join(BASE_PATH, 'train_1.csv')
TEST_CSV   = os.path.join(BASE_PATH, 'test.csv')
TRAIN_DIR  = os.path.join(BASE_PATH, 'train_images', 'train_images')
TEST_DIR   = os.path.join(BASE_PATH, 'test_images',  'test_images')

# ── Nommage clinique des classes ─────────────────────────────
CLASS_NAMES = {
    0: 'No DR',
    1: 'Mild DR',
    2: 'Moderate DR',
    3: 'Severe DR',
    4: 'Proliferative DR'
}
COLORS = ['#27ae60', '#f39c12', '#e67e22', '#c0392b', '#8e44ad']

# Chargement et audit basique

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print("╔══════════════════════════════════════╗")
print("║     AUDIT DU DATASET APTOS 2019      ║")
print("╚══════════════════════════════════════╝\n")

print(f"  Lignes train     : {len(df_train)}")
print(f"  Lignes test      : {len(df_test)}")
print(f"  Colonnes train   : {list(df_train.columns)}")
print(f"  Valeurs manquan. : {df_train.isnull().sum().to_dict()}")
print(f"  Doublons         : {df_train.duplicated().sum()}")
print(f"\nAperçu :")
display(df_train.head(8))

print(f"\nRépartition par classe :")
for label, count in df_train['diagnosis'].value_counts().sort_index().items():
    pct = count / len(df_train) * 100
    bar = '█' * int(pct / 2)
    print(f"  Classe {label} ({CLASS_NAMES[label]:18s}) : {count:4d}  {bar} {pct:.1f}%")

# Visualisation de la distribution des classes

In [ ]:
class_counts = df_train['diagnosis'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution des classes — APTOS 2019', fontsize=13, fontweight='bold')

# ── Barplot avec annotations ─────────────────────────────────
bars = axes[0].bar(
    [CLASS_NAMES[i] for i in class_counts.index],
    class_counts.values,
    color=COLORS, edgecolor='black', linewidth=0.6
)
axes[0].set_title('Nombre d\'images par stade DR')
axes[0].set_ylabel('Nombre d\'images')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylim(0, class_counts.max() * 1.15)

for bar, val in zip(bars, class_counts.values):
    pct = val / len(df_train) * 100
    axes[0].text(
        bar.get_x() + bar.get_width() / 2., bar.get_height() + 20,
        f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

# Ligne de référence : distribution équilibrée idéale
balanced = len(df_train) / 5
axes[0].axhline(balanced, color='red', linestyle='--', linewidth=1, label=f'Distribution équilibrée ({int(balanced)})')
axes[0].legend(fontsize=9)

# ── Pie chart ────────────────────────────────────────────────
wedges, texts, autotexts = axes[1].pie(
    class_counts.values,
    labels=[CLASS_NAMES[i] for i in class_counts.index],
    autopct='%1.1f%%', colors=COLORS,
    startangle=90, pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title('Répartition proportionnelle')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Quantification du déséquilibre ──────────────────────────
print("\n=== ANALYSE DU DÉSÉQUILIBRE DE CLASSES ===")
majority = class_counts.max()
for label, count in class_counts.items():
    ratio = majority / count
    status = "⚠️  minoritaire" if ratio > 3 else "✓"
    print(f"  Classe {label} ({CLASS_NAMES[label]:18s}) : ratio {ratio:5.1f}x {status}")

# Visualisation des images par classe

In [ ]:
def load_image(img_id, directory, size=(256, 256)):
    """Charge une image en RGB avec fallback png/jpeg."""
    for ext in ['.png', '.jpeg', '.jpg']:
        path = os.path.join(directory, img_id + ext)
        if os.path.exists(path):
            img = cv2.imread(path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                return cv2.resize(img, size)
    return None

N_SAMPLES = 5  # images par classe

fig, axes = plt.subplots(5, N_SAMPLES, figsize=(N_SAMPLES * 3, 5 * 3))
fig.suptitle('Exemples d\'images rétiniennes par stade de rétinopathie diabétique\n(Dataset APTOS 2019)',
             fontsize=12, fontweight='bold', y=1.01)

for label in range(5):
    samples = df_train[df_train['diagnosis'] == label].sample(N_SAMPLES, random_state=42)
    for j, (_, row) in enumerate(samples.iterrows()):
        img = load_image(row['id_code'], TRAIN_DIR)
        if img is not None:
            axes[label][j].imshow(img)
        axes[label][j].axis('off')
        if j == 0:
            axes[label][j].set_ylabel(
                f"Classe {label}\n{CLASS_NAMES[label]}",
                fontsize=10, fontweight='bold',
                rotation=0, labelpad=90, va='center',
                color=COLORS[label]
            )

plt.tight_layout()
plt.savefig('eda_images_par_classe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
!git clone https://github.com/haliait/sys_retinopathie_diabetique

In [ ]:
%cd sys_retinopathie_diabetique

In [ ]:
!git add .
!git commit -m "init phase 1"

In [ ]:
!git remote -v

In [ ]:
!git config --global user.name "SaraTaha9"
!git config --global user.email "tsara6317@example.com"

In [ ]:
!git push origin main

In [ ]:
!git add .
!git commit -m "init Phase 1"

In [ ]:
!git status

In [ ]:
!git diff

In [ ]:
!git branch

In [ ]:
!git remote -v

In [ ]:
!git log --oneline -5

In [ ]:
!git status

In [ ]:
!ls

In [ ]:
!mv /content/DR.ipynb .

# Analyse des dimensions et de la qualité des images

In [ ]:
print("=== ANALYSE DES PROPRIÉTÉS DES IMAGES ===\n")

# ── Scan d'un sous-ensemble pour la rapidité ────────────────
SAMPLE_SIZE = 300
sample_ids = df_train['id_code'].sample(SAMPLE_SIZE, random_state=42).values

dimensions, pixel_means, dark_ids, aspect_ratios = [], [], [], []

for img_id in sample_ids:
    for ext in ['.png', '.jpeg', '.jpg']:
        path = os.path.join(TRAIN_DIR, img_id + ext)
        if os.path.exists(path):
            try:
                with Image.open(path) as pil_img:
                    w, h = pil_img.size
                    dimensions.append((w, h))
                    aspect_ratios.append(w / h)

                img_cv = cv2.imread(path)
                img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
                mean_val = np.mean(img_rgb)
                pixel_means.append(mean_val)
                if mean_val < 30:
                    dark_ids.append(img_id)
            except Exception as e:
                pass
            break

widths  = [d[0] for d in dimensions]
heights = [d[1] for d in dimensions]

print(f"[DIMENSIONS] (sur {len(dimensions)} images)")
print(f"  Largeur  — min: {min(widths)}, max: {max(widths)}, moy: {np.mean(widths):.0f} px")
print(f"  Hauteur  — min: {min(heights)}, max: {max(heights)}, moy: {np.mean(heights):.0f} px")
print(f"  Ratio H/W — min: {min(aspect_ratios):.2f}, max: {max(aspect_ratios):.2f}, moy: {np.mean(aspect_ratios):.2f}")

print(f"\n[QUALITÉ]")
print(f"  Intensité moyenne — min: {min(pixel_means):.1f}, max: {max(pixel_means):.1f}, moy: {np.mean(pixel_means):.1f}")
print(f"  Images très sombres (moy < 30) : {len(dark_ids)} ({len(dark_ids)/SAMPLE_SIZE*100:.1f}%)")

top_dims = Counter(dimensions).most_common(8)
print(f"\n[TOP 8 RÉSOLUTIONS]")
for dim, cnt in top_dims:
    print(f"  {dim[0]:4d} × {dim[1]:4d} : {cnt:3d} images")

# ── Visualisation ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Propriétés des images APTOS 2019', fontsize=12, fontweight='bold')

axes[0].hist(widths, bins=25, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('Distribution des largeurs')
axes[0].set_xlabel('Pixels')
axes[0].set_ylabel('Nombre d\'images')

axes[1].hist(pixel_means, bins=25, color='coral', edgecolor='black', linewidth=0.5)
axes[1].axvline(30, color='red', linestyle='--', label='Seuil images sombres (30)')
axes[1].set_title('Distribution de l\'intensité moyenne')
axes[1].set_xlabel('Intensité moyenne (0–255)')
axes[1].legend(fontsize=8)

axes[2].scatter(widths, heights, alpha=0.4, color='purple', s=15)
axes[2].set_title('Largeur vs Hauteur')
axes[2].set_xlabel('Largeur (px)')
axes[2].set_ylabel('Hauteur (px)')

plt.tight_layout()
plt.savefig('eda_image_properties.png', dpi=150, bbox_inches='tight')
plt.show()